In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as plt
import os 
import datetime

DATE = datetime.date.today().strftime("%Y_%m_%d")
YEAR = datetime.date.today().year
MRI_YEAR_TOTAL_AMBITION =2019
TARGET_YEAR_TOTAL_AMBITION = 2030

#### Now I'll calculate the annualised ambition and the total ambition for PBL and CAT datasets
##### For this first regression I'll only take into account the cities with PLEDGES and the unconditional NDC's which will be an average between the min and the max

In [2]:
def read_files_pbl(file="Infographics PBL NDC Tool March 2024.xlsx"):
    path = r"C:\Users\Marti183\CAMDA_NSA\climate_ambition"
    newpath = os.path.join(path, file)
    df = pd.read_excel(newpath, header=None)
    first_header = df.iloc[0]
    second_header = df.iloc[1]

    #Add first header to df.columns
    new_cols = []
    for a, b in zip(first_header, second_header):
        #if name.notna(): #This wont work because notana() only works for pd.Series
        if pd.notna(a):
           new_cols.append(f"{a}_{b}")
        else:
           new_cols.append(b)
    
    df.columns = new_cols
   
    #assign dataset to the correct rows
    df = df.iloc[2:].reset_index()
    
    return df

In [3]:
#This function calculates the annualised ambition of the data from the pbl tool
#The target year of this data is 2030
def get_pbl_annualised_ambition():
    data = read_files_pbl()
    
    #Filter for pledging countries
    data = data[data["Pledge/action"] == "Pledge"]

    #EXCLUDE LULUCF where possible however if I exclude LULUCF in this case will leave with just a couple of countries
    
    #Calculate target year emissions as the average between min and max unconditional 
    data["emissions_target_year"] = (data["2030 NDCs_Unconditional INDCs, Min"] + data["Unconditional INDCs, Max"])/2
    target_year = 2030
    #get the emissions of the MRI year 
    MRI_year = 2021
    #calculate annualised ambition, in the end I only need geometric ambition
    #data["annualised_arithmetic_ambition"] = (data[MRI_year]-data["emissions_target_year"])/(data[MRI_year] *(target_year - MRI_year))
    data["annualised_geometric_ambition"] = (1 + (data[MRI_year]-data["emissions_target_year"])/data[MRI_year]) ** (1/(target_year - MRI_year)) -1

    #calculate total ambition as the sum between the annualised ambition in the period between the target and the MRI years.
    #To calculate absolute ambition we have fixded the MRI_year to 2019 and the target year to 2030, SO I THINK HERE I SHOULD ALSO USE 2019 AS IN CLIMACTOR I WILL USE 2019 #TODO
    #data["total_arithmetic_ambition"] = data["annualised_arithmetic_ambition"]*(target_year-MRI_year)
    data["total_ambition"] = data[MRI_YEAR_TOTAL_AMBITION] - data["emissions_target_year"]


    #return a dataset with only interested columns 
    data = pd.concat([data.iloc[:, 4:9], data.iloc[:, -3:]], axis=1)
    data["Unit"] = "Mt CO2 equivalent"
    data = data.rename(columns={"(Mt CO2 equivalent)_Country name" : "Country_name"})

    #I just realised there are two ISO columns one tih the actual ISO and one with just random numbers
    #The column I want to delte is in the position 1 
    #data = data.drop(data.columns[1], axis=1) This does not work because drop works with LABELS so whem Im saying drop data.columns[1] it means drop columns named ISO
    data =data.iloc[:, [i for i in range(data.shape[1]) if i != 1]]

    data.to_csv(f"pbl_annualised_ambition_{DATE}.csv")
    return data


In [4]:
pbl_annualised_ambition =get_pbl_annualised_ambition()

In [5]:
pbl_annualised_ambition

,Country_name,VSF nr,ISO,LULUCF,emissions_target_year,annualised_geometric_ambition,total_ambition,Unit
26,Brazil,22,BRA,incl,1224.008138,0.031147,482.531855,Mt CO2 equivalent
33,Canada,33,CAN,excl,424.764,0.035298,298.91529,Mt CO2 equivalent
38,Chile,39,CHL,excl,95.0,0.011166,13.197756,Mt CO2 equivalent
39,China,40,CHN,incl,13744.651933,-0.006163,-1497.480392,Mt CO2 equivalent
45,Costa Rica,47,CRI,incl,9.11,0.011917,3.40437,Mt CO2 equivalent
88,India,90,IND,incl,4445.399503,-0.068058,-1511.854583,Mt CO2 equivalent
89,Indonesia,91,IDN,incl,1954.0759,-0.007295,-76.265201,Mt CO2 equivalent
93,Israel,95,ISR,excl,58.0,0.030963,27.636092,Mt CO2 equivalent
96,Japan,99,JPN,incl,808.02,0.027471,349.022484,Mt CO2 equivalent
102,South Korea,104,KOR,incl,436.56,0.031413,234.140295,Mt CO2 equivalent


In [6]:
def climate_action_tracker(file="CAT_06052026_CountryAssessmentData_DataExplorer_1.xlsx"):
    path = r"C:\Users\Marti183\CAMDA_NSA\climate_ambition"
    newpath = os.path.join(path, file)
    df = pd.read_excel(newpath, header=18)
    df = df.drop(columns= ["Unnamed: 0", "Unnamed: 1"])
    return df


In [7]:
climate_action_tracker = climate_action_tracker()

In [8]:
# This function gets the most recent inventory year of a dataset
# This year will be the year before an empty cell in the columns so 1990-2022 all have numeric values but 2023 is empty. Then 2022 is the MRI year

def get_MRI_year(data):
    data = data[data["Scenario"] == "Historical"]
    #data = data.select_dtypes(include=np.number)
    data = data[data["Sector"] == "Economy-wide, excluding LULUCF"]
    #MRI_year = data.isna().idxmax(axis=1, skipna=False) #idxmax() on h.isna() gives the first column where the row is missing
    MRI_year = data.apply(lambda x: x.last_valid_index(), axis=1)#last_valid_index() gives the last column where the row is still 
    #get the emissions
    #assign country to the MRI year
    countries = data.loc[MRI_year.index, "Country"]
    MRI_year_c = pd.Series(MRI_year.values, index = countries.values)

    data["MRI_year"] = data.Country.map(MRI_year_c)
    #Here Im telling pandas from x row select the value from the year = x["check"] so for ex x[2022]
    MRI_emissions = data.apply(lambda x: x[x["MRI_year"]], axis=1)
    if MRI_year.index.equals(MRI_emissions.index):
        MRI_emissions_c = pd.Series(MRI_emissions.values, index = countries.values)
    
    else:
        countries = data.loc[MRI_emissions.index, "Country"]
        MRI_emissions_c = pd.Series(MRI_emissions.values, index = countries.values)
   
    return MRI_year_c, MRI_emissions_c
    

In [9]:
#This fucntion gets the target emissions and year for each scenario

def get_target_emissions(data):
    # Scenarios we want to keep and process
    scenarios = ["NDC Unconditional, Max", "Current Policy, Max", "Current Policy, Min", "NDC Unconditional, Min", "NDC Conditional, Min", "NDC Conditional, Max"]

    # Keep only economy-wide rows and only the scenarios we care about
    filter_data = data[data["Sector"] == "Economy-wide, excluding LULUCF"]
    filter_data = filter_data[filter_data["Scenario"].isin(scenarios)]

    #
    copy_data = filter_data.copy()

    #Process one scenario at a time
    for scenario in scenarios: 
        #FIRST STEP: GET THE YEAR
        #first I have to select the row of the scenario im evaluating (Remember that in CAT the scenarios are in rows)
        use_data = copy_data[copy_data["Scenario"] == scenario]
        # Keep only the year/emissions columns, not text columns like Country or Scenario
        work_data = use_data.select_dtypes(include=np.number)

        # Name of the new column that will store the most recent inventory year
        scenario_year = f"{scenario}_year"
        # Find the last year with a numeric value in each row
        y = work_data.apply(lambda x: x.last_valid_index(), axis=1)
        countries = use_data.loc[y.index, "Country"]
        yc = pd.Series(y.values, index=countries.values)
        # Store the year in the main filtered table
        filter_data[scenario_year] = filter_data.Country.map(yc)

        #SECOND STEP: GET THE EMISSIONS
        #Remember that in CAT the scenarios are in rows so to match the correct emissions I need the specific row of the scenario im evaulating!!!
        use_data[scenario_year] = use_data.Country.map(yc)
        # Name of the new column that will store the emissions value for that year
        scenario_emissions = f"{scenario}_emissions"
        # For each row, use the year stored in scenario_year to pick the matching year column
        e = use_data.apply(lambda x: x[x[scenario_year]], axis=1)
        # Match the emissions values back to the correct countries
        countries = use_data.loc[e.index, "Country"]
        ec= pd.Series(e.values, index=countries.values)
        # Store the emissions in the main filtered table
        filter_data[scenario_emissions] = filter_data.Country.map(ec)

        #THRID STEP: GET THE EMISSIONS FOR TOTAL AMBITION
        series = pd.Series(use_data[TARGET_YEAR_TOTAL_AMBITION].values, index=use_data["Country"].values)
        filter_data[f"{TARGET_YEAR_TOTAL_AMBITION}_{scenario_emissions}"] = filter_data.Country.map(series)


    # Remove the year columns and other extra columns we don't need in the final output
    drop_columns = [ i for i in range(1990, 2036)]
    drop_columns = drop_columns + ["Scenario"]
    df = filter_data.drop(columns=drop_columns)

    # Remove duplicate rows from the main dataset 
    df = df.drop_duplicates()
    return df

In [10]:
#This fucntion is to get the emissions for MRI_YEAR_TOTAL_AMBITION =2019
def get_MRI_year_total_ambition(data):
    data = data[data["Scenario"] == "Historical"]
    data = data[data["Sector"] == "Economy-wide, excluding LULUCF"]
    emissions = pd.Series(data[MRI_YEAR_TOTAL_AMBITION].values, index=data["Country"].values)
    return emissions

In [11]:
def get_cat_annualised_ambition(cat_data):
    scenarios = ["NDC Unconditional, Max", "Current Policy, Max", "Current Policy, Min", "NDC Unconditional, Min", "NDC Conditional, Min", "NDC Conditional, Max"]
    data = get_target_emissions(cat_data)

    #First I need to calculate the average between the mininimum and maximum values of the three different scenarios

    pro_scenarios = ["NDC Unconditional", "NDC Conditional", "Current Policy"]
    for scenario in pro_scenarios:
        av = [x for x in scenarios if scenario in x]
        new_av = [x + "_emissions" for x in av]
        data[f"{scenario}_emissions"] = data[new_av].mean(axis=1)

        #Now check if the target years are the same for both min and max scenarios
        #if (data[f"{scenario}, Max_year"] == data[f"{scenario}, Min_year"]).all():
        #That previous line of code did not work because for one country they only have values for one scenario
        data[f"{scenario}_year"] = np.where(data[f"{scenario}, Max_year"] == data[f"{scenario}, Min_year"], data[f"{scenario}, Max_year"], np.maximum(data[f"{scenario}, Max_year"], data[f"{scenario}, Min_year"]))
        
        #Calculate the average between min and max 2030 scenario emissions
        data[f"{TARGET_YEAR_TOTAL_AMBITION}_{scenario}"] = data[[f"{TARGET_YEAR_TOTAL_AMBITION}_{scenario}, Min_emissions", f"{TARGET_YEAR_TOTAL_AMBITION}_{scenario}, Max_emissions"]].mean(axis=1)

    #clean dataset
    drop_cols = [col for col in data.columns if ("Min" in col) or ("Max" in col)]
    data = data.drop(columns=drop_cols)
    
    # Get the historical value for most recent inventory year emissions for each country.
    MRI_year_c, MRI_emissions_c = get_MRI_year(cat_data)

    #get the historical value for MRI YEAR TOTAL AMBITION emissions
    MRI_total_ambition_emissions =get_MRI_year_total_ambition(cat_data)

    #assign country to the MRI year and emissions value
    data["MRI_year"] = data.Country.map(MRI_year_c)
    data["MRI_emissions"] = data.Country.map(MRI_emissions_c)

    #assign country to MRI YEAR TOTAL AMBITION emissions
    data[f"{MRI_YEAR_TOTAL_AMBITION}_emissions"] = data.Country.map(MRI_total_ambition_emissions)
  
    #calculate annualised ambition
    for scenario in pro_scenarios: 
        #arithmetic
        #data[f"{scenario}_arithmetic_ambition"] = (data["MRI_emissions"] - data[f"{scenario}_emissions"])/(data["MRI_emissions"] *(data[f"{scenario}_year"] - data["MRI_year"]))
        #geometric
        data[f"{scenario}_annualised_geometric_ambition"] = (1 + (data["MRI_emissions"] - data[f"{scenario}_emissions"])/data["MRI_emissions"]) ** (1/(data[f"{scenario}_year"] - data["MRI_year"])) -1 

    #calculate total ambition
    #For climate action tracker I know that the minimum most recent inventory year is 2020 so 2019 is safe

        data[f"{scenario}_total_ambition"] = data[f"{MRI_YEAR_TOTAL_AMBITION}_emissions"] - data[f"{TARGET_YEAR_TOTAL_AMBITION}_{scenario}"]


    data.to_csv(f"CAT_annualised_ambition.{DATE}.csv")
    return data
    

In [12]:
cat_ambition = get_cat_annualised_ambition(climate_action_tracker)


In [13]:
cat_ambition

,Version,Country,Sector,Indicator,Unit,NDC Unconditional_emissions,NDC Unconditional_year,2030_NDC Unconditional,NDC Conditional_emissions,NDC Conditional_year,...,2030_Current Policy,MRI_year,MRI_emissions,2019_emissions,NDC Unconditional_annualised_geometric_ambition,NDC Unconditional_total_ambition,NDC Conditional_annualised_geometric_ambition,NDC Conditional_total_ambition,Current Policy_annualised_geometric_ambition,Current Policy_total_ambition
0,112024,ARE,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR4),185.150,2030.0,185.150,NaN,NaN,...,236.205000,2022,215.720000,199.740000,0.016704,14.590000,NaN,NaN,-0.013421,-36.465000
14,92024,ARG,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR4),350.930,2030.0,350.930,NaN,NaN,...,405.040000,2022,352.840000,335.970000,0.000675,-14.960000,NaN,NaN,-0.021153,-69.070000
28,92025,AUS,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR5),268.895,2035.0,412.410,NaN,NaN,...,430.350000,2024,520.220000,557.280000,0.036481,144.870000,NaN,NaN,0.023296,126.930000
42,82024,BRA,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR4),883.870,2030.0,883.870,NaN,NaN,...,1171.050000,2022,1170.450000,1112.170000,0.027755,228.300000,NaN,NaN,0.001609,-58.880000
56,72023,BTN,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR4),3.075,2030.0,3.075,1.865,2030.0,...,2.925000,2022,2.100000,2.560000,-0.075053,-0.515000,0.013348,0.695000,-0.060468,-0.365000
74,112025,CAN,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR5),454.180,2030.0,454.180,NaN,NaN,...,643.900000,2023,694.000000,747.360000,0.043313,293.180000,NaN,NaN,0.007776,103.460000
88,122024,CHE,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR4),27.010,2030.0,27.010,NaN,NaN,...,28.120000,2022,41.410000,46.310000,0.038008,19.300000,NaN,NaN,0.037157,18.190000
104,92025,CHL,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR5),90.000,2035.0,95.000,NaN,NaN,...,105.025000,2023,109.860000,112.780000,0.013944,17.780000,NaN,NaN,0.006368,7.755000
118,92025,CHN,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR5),14154.080,2035.0,14841.840,NaN,NaN,...,14961.320000,2023,15008.100000,13702.300000,0.004623,-1139.540000,NaN,NaN,0.009374,-1259.020000
132,102025,COL,"Economy-wide, excluding LULUCF",Emissions (absolute),MtCO2e (GWP-AR5),139.650,2035.0,151.090,NaN,NaN,...,206.910000,2023,198.550000,183.820000,0.021885,32.730000,NaN,NaN,-0.002841,-23.090000


#### Upload and calculation of total ambition of climactor files
##### Citys-only!!!

In [14]:
def read_files_climactor(data):
    d = pd.read_csv(data, header=0)
    return d

In [15]:
#Given a series of numbers numbers this fuction calculates the closest to 2030
def closer_to_2030(sernum):
    diff = [abs(x - 2030) for x in sernum ]
    i = diff.index(min(diff))
    return sernum.iloc[i]

In [16]:
#Total ambition is defined as the emissions in (fixed iventory year =2019) - target year emissions (fixed at 2030)
# Merge with Yuetongs annualised ambition dataset

def get_total_ambition():
    #load both datasets
    climactor_raw = read_files_climactor("climactor_targets_combined.csv")
    climactor_annualised_ambition = read_files_climactor("climactor_annualized_ambition.csv")

    #dataset cleaning
    #Only include cities
    climactor_raw = climactor_raw[climactor_raw["entity_type"] == "City"]
    #Ignore entries without a GDAM_id
    climactor_raw = climactor_raw[climactor_raw["GDAM_id"].notna()]
    #only include entries with baseline value and year
    climactor_raw = climactor_raw[(climactor_raw["baseline_value"].notna()) & (climactor_raw["baseline_year"].notna())]
    #Filter out cities with targets before 2026
    climactor_raw = climactor_raw[climactor_raw["target_year"] >= YEAR]

    #Only include relevant columns
    #As Yuetong calaculates annualised mabitioon differently im gonna calculate it myself
    climactor_annualised_ambition = climactor_annualised_ambition[["GDAM_id", "target_year_emission"#, "annualized_ambition"
                                                                   ]]

    #There are entries repeated with for example different inventory years or target years 
    #filter to include inv year = np.nan or for rows with multiple inventory years only the most recent one
    mask = (
    climactor_raw["inv_year"].isna()
    | climactor_raw["inv_year"].eq(
        climactor_raw.groupby("GDAM_id")["inv_year"].transform("max"))
        )
    climactor_fil = climactor_raw[mask]
    
    #filter for target closer to 2030 for cities with several target years. 
    mask2 = climactor_fil["target_year"].eq(climactor_fil.groupby("GDAM_id")["target_year"].transform(closer_to_2030))
    data = climactor_fil[mask2]
    
    # Calculate target year emissions
    data["target_year_emission"] = data["baseline_value"]* (1-data["target_value"]/100) # Target value is a percentage

    #Some countries have already reached their targets (total_emissions < target year emissions) so we dont have to include them in our analysis
    data = data[data["total_emissions"] > data["target_year_emission"]]

    #There are some entries with same inventory year but different total_emissions (maybe data collected from different sources)
    #In this case, we discussed to calcualte the ambition using the baseline value
    #quick check to verify that all these entries have the same baseline ambition so we can only keep one unique row per city

    check2 = (data.groupby("GDAM_id").agg(
        n_total_emissions=("total_emissions", "nunique"),
        n_baseline_value=("baseline_value", "nunique")
    ))

    result = check2[check2["n_total_emissions"] > 1]
    if (result["n_baseline_value"] == 1).all():
        print("Every entry with different total emissions for the same inventory year has the same baseline value")
    
    else:
        raise ValueError(" NOT every entry with different total emissions for the same inventory year has the same baseline value")
    
    #list of all the GDAM_ids with this ambiguity
    #"GDAM_id is not a column anymore is the index!!"
    result = result.reset_index()
    ambiguous_cities = set(result["GDAM_id"]) 

    #For this entries I will calculated the annualised ambition from the baseline value
    #Calculate the annualised ambition from the baseline year for years with an inventory year greater than 2019, np.nan OR SEVERAL TOTAL EMISSISIONS-- INTERMEDIARY STEP
    data["annualised_ambition_from_baseline"] = (1 + (data["baseline_value"] - data["target_year_emission"])/data["baseline_value"]) ** (1/(data["target_year"] - data["baseline_year"])) -1 

    #calculate annualised ambition 
    data["annualised_ambition"] = np.where(data["inv_year"].isna() | data["GDAM_id"].isin(ambiguous_cities), data["annualised_ambition_from_baseline"], 
                                           (1 + (data["total_emissions"] - data["target_year_emission"])/data["total_emissions"]) ** (1/(data["target_year"] - data["inv_year"])) -1 )

    # #Let's check that Yuetong also used annualised ambition from baseline for missing inv years
    # check1 = data[data["inv_year"].isna()]
    # if (check1["annualized_ambition"] == check1["annualised_ambition_from_baseline"]).all(skipna=True):
    #     print("Yuetong also used annualised ambtion from baseline for missing inv_year values")
    # else:
    #     raise ValueError("Yuetong and I used different methods to calculate annualised ambition for missing inv_year values")
    
    #quick check that all target years are greater than the MRI years
    check =  data[data["inv_year"].notna()]
    if (check["target_year"] > check["inv_year"]).all():
        print("Target years are all greater than MRI years")
    else:
        raise ValueError("Not all target years are greater than the MRI years")


    #Now I need to calculate emisions for the fixed MRI year = 2019
    #There are THREE POSSIBILITIES, INV_YEAR == 2019, INV_YEAR > 2019, (INV_YEAR < 2019 or INV_YEAR == NaN, GDAM_id is in ambiguous cities) these last three fall under the same category as the emissions in this case are calculated using the baseline
    data[f"{MRI_YEAR_TOTAL_AMBITION}_emissions_for_total_ambition"] = np.where(data["inv_year"] > MRI_YEAR_TOTAL_AMBITION | data["GDAM_id"].isin(ambiguous_cities) | data["inv_year"].isna(),
                                                                        data["baseline_value"]*(1-data["annualised_ambition_from_baseline"])**(MRI_YEAR_TOTAL_AMBITION-data["baseline_year"]),
                                                                        np.where(data["inv_year"] < MRI_YEAR_TOTAL_AMBITION, data["total_emissions"]*(1-data["annualised_ambition"])**(MRI_YEAR_TOTAL_AMBITION-data["inv_year"]), 
                                                                         data["total_emissions"] ))
    
    #Now I need to calculate target emissions 
    #There are THREE POSSIBILITIES: TARGET_YEAR == 2030, TARGET_YEAR != 2030 & INV_YEAR not NaN, TARGET_YEAR != 2030 & INV_YEAR == NaN | GDAM_id in ambiguopus cities
    data[f"{TARGET_YEAR_TOTAL_AMBITION}_emissions"] = np.where(data["target_year"] ==  TARGET_YEAR_TOTAL_AMBITION, data["target_year_emission"], 
                                                     np.where(data["inv_year"].isna() | data["GDAM_id"].isin(ambiguous_cities), 
                                                       data["baseline_value"]*(1-data["annualised_ambition_from_baseline"])**(TARGET_YEAR_TOTAL_AMBITION-data["baseline_year"] ), 
                                                    data["total_emissions"]*(1-data["annualised_ambition"])**(TARGET_YEAR_TOTAL_AMBITION-data["inv_year"])))


    #Calculation of ANSOLUTE ambition
    data["absolute_ambition"] = data[f"{MRI_YEAR_TOTAL_AMBITION}_emissions_for_total_ambition"] - data[f"{TARGET_YEAR_TOTAL_AMBITION}_emissions"]

    #Keep only one row per city
    data = data.drop(columns="total_emissions")
    data = data.drop_duplicates()

    #save dataframe
    data.to_csv(f"ClimActor_ambition.{DATE}.csv")

    return data


In [17]:
climactor_ambition = get_total_ambition()


Every entry with different total emissions for the same inventory year has the same baseline value
Target years are all greater than MRI years


In [18]:
climactor_raw = read_files_climactor("climactor_targets_combined.csv")
climactor_raw = climactor_raw[climactor_raw["entity_type"] == "City"]
climactor_raw = climactor_raw[climactor_raw["GDAM_id"].notna()]
climactor_raw = climactor_raw[(climactor_raw["baseline_value"].notna()) & (climactor_raw["baseline_year"].notna())]
print(climactor_raw)
mask = (
    climactor_raw["inv_year"].isna()
    | climactor_raw["inv_year"].eq(
        climactor_raw.groupby("GDAM_id")["inv_year"].transform("max")
    )
)
l = climactor_raw[mask]
print(l)
mask2 = l["target_year"].eq(l.groupby("GDAM_id")["target_year"].transform("max"))
z = l[mask2]
z

      iso entity_type      climactor_id  baseline_value  baseline_year  \
0     BEL        City  0033fb16f68dfeeb       2629824.0           2005   
1     BEL        City  0033fb16f68dfeeb       2629824.0           2005   
3     BEL        City  00732a73f4f159b3         43491.3           2011   
4     BEL        City  00732a73f4f159b3         43491.3           2011   
5     ESP        City  00752000db04c515         11423.0           2005   
...   ...         ...               ...             ...            ...   
9398  BEL        City  ff904d7013b2688c         75769.0           2011   
9399  BEL        City  ff904d7013b2688c         75769.0           2011   
9400  BEL        City  ff904d7013b2688c         75769.0           2011   
9403  ITA        City  ffa60dd141d5c943         83747.8           2010   
9410  ESP        City  ffe0efb6be3172e5          3489.4           2005   

      inv_year  total_emissions  target_year  target_value  \
0          NaN              NaN         2030     

,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,total_emissions,target_year,target_value,target_record,GDAM_id
1,BEL,City,0033fb16f68dfeeb,2629824.000,2005,NaN,NaN,2050,100.00,0033fb16f68dfeeb_2050,BEL.2.1.1.2_1
4,BEL,City,00732a73f4f159b3,43491.300,2011,2014.0,25487.30,2030,40.00,00732a73f4f159b3_2030,BEL.2.2.1.5_1
6,ESP,City,00752000db04c515,11423.000,2005,2014.0,8726.40,2030,40.00,00752000db04c515_2030,ESP.09.08.08222
7,ITA,City,0083ef4ed71b869e,7739.785,2011,2011.0,7739.80,2030,40.00,0083ef4ed71b869e_2030,ITA.19.86.003
10,BEL,City,00887dae5af7b824,48759.200,2011,2020.0,45265.40,2030,40.00,00887dae5af7b824_2030,BEL.2.1.3.22_1
...,...,...,...,...,...,...,...,...,...,...,...
9387,USA,City,ff4e338346fcdebb,2597305.150,2019,2019.0,2597305.15,2050,100.00,ff4e338346fcdebb_2050,USA.16.77.1600000US1921000
9393,ITA,City,ff8d794fe84c1e9d,22806.700,2011,2011.0,22806.70,2050,40.42,ff8d794fe84c1e9d_2050,ITA.19.87.047
9400,BEL,City,ff904d7013b2688c,75769.000,2011,2020.0,67296.70,2030,40.00,ff904d7013b2688c_2030,BEL.2.2.3.1_1
9403,ITA,City,ffa60dd141d5c943,83747.800,2010,2010.0,83747.80,2030,40.00,ffa60dd141d5c943_2030,ITA.15.63.073


In [19]:
(z["target_year"] > z["inv_year"]).all(skipna=True)

np.False_

In [20]:
climactor_raw = climactor_raw[climactor_raw["target_year"] > 2026]
climactor_raw.groupby("GDAM_id")["target_year"].transform(closer_to_2030)

0       2030
1       2030
3       2030
4       2030
5       2030
        ... 
9398    2030
9399    2030
9400    2030
9403    2030
9410    2030
Name: target_year, Length: 5157, dtype: int64

In [21]:
climactor_raw

,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,total_emissions,target_year,target_value,target_record,GDAM_id
0,BEL,City,0033fb16f68dfeeb,2629824.0,2005,NaN,NaN,2030,50.0,0033fb16f68dfeeb_2030,BEL.2.1.1.2_1
1,BEL,City,0033fb16f68dfeeb,2629824.0,2005,NaN,NaN,2050,100.0,0033fb16f68dfeeb_2050,BEL.2.1.1.2_1
3,BEL,City,00732a73f4f159b3,43491.3,2011,2011.0,43491.3,2030,40.0,00732a73f4f159b3_2030,BEL.2.2.1.5_1
4,BEL,City,00732a73f4f159b3,43491.3,2011,2014.0,25487.3,2030,40.0,00732a73f4f159b3_2030,BEL.2.2.1.5_1
5,ESP,City,00752000db04c515,11423.0,2005,2005.0,10036.9,2030,40.0,00752000db04c515_2030,ESP.09.08.08222
...,...,...,...,...,...,...,...,...,...,...,...
9398,BEL,City,ff904d7013b2688c,75769.0,2011,2011.0,75769.0,2030,40.0,ff904d7013b2688c_2030,BEL.2.2.3.1_1
9399,BEL,City,ff904d7013b2688c,75769.0,2011,2014.0,76492.6,2030,40.0,ff904d7013b2688c_2030,BEL.2.2.3.1_1
9400,BEL,City,ff904d7013b2688c,75769.0,2011,2020.0,67296.7,2030,40.0,ff904d7013b2688c_2030,BEL.2.2.3.1_1
9403,ITA,City,ffa60dd141d5c943,83747.8,2010,2010.0,83747.8,2030,40.0,ffa60dd141d5c943_2030,ITA.15.63.073


In [22]:
climactor_annualised_ambition = read_files_climactor("climactor_annualized_ambition.csv")

In [23]:
climactor_annualised_ambition.loc[climactor_annualised_ambition["GDAM_id"]== "ITA.19.87.047", :	]

,iso,climactor_id,GDAM_id,baseline_value,baseline_year,target_year,target_value,target_year_emission,annualized_ambition
3650,ITA,ff8d794fe84c1e9d,ITA.19.87.047,22806.7,2011,2050,40.42,13588.23186,0.008742


In [24]:
climactor_raw = read_files_climactor("climactor_targets_combined.csv")
p = climactor_raw.loc[climactor_raw["GDAM_id"] == "ITA.19.87.047", :]
p

,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,total_emissions,target_year,target_value,target_record,GDAM_id
9393,ITA,City,ff8d794fe84c1e9d,22806.7,2011,2011.0,22806.7,2050,40.42,ff8d794fe84c1e9d_2050,ITA.19.87.047


In [25]:
a = z[z["inv_year"].notna()]
a

,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,total_emissions,target_year,target_value,target_record,GDAM_id
4,BEL,City,00732a73f4f159b3,43491.300,2011,2014.0,25487.30,2030,40.00,00732a73f4f159b3_2030,BEL.2.2.1.5_1
6,ESP,City,00752000db04c515,11423.000,2005,2014.0,8726.40,2030,40.00,00752000db04c515_2030,ESP.09.08.08222
7,ITA,City,0083ef4ed71b869e,7739.785,2011,2011.0,7739.80,2030,40.00,0083ef4ed71b869e_2030,ITA.19.86.003
10,BEL,City,00887dae5af7b824,48759.200,2011,2020.0,45265.40,2030,40.00,00887dae5af7b824_2030,BEL.2.1.3.22_1
13,ITA,City,00b3f9e9c6778784,8965.400,2011,2011.0,8965.40,2030,40.00,00b3f9e9c6778784_2030,ITA.19.84.007
...,...,...,...,...,...,...,...,...,...,...,...
9387,USA,City,ff4e338346fcdebb,2597305.150,2019,2019.0,2597305.15,2050,100.00,ff4e338346fcdebb_2050,USA.16.77.1600000US1921000
9393,ITA,City,ff8d794fe84c1e9d,22806.700,2011,2011.0,22806.70,2050,40.42,ff8d794fe84c1e9d_2050,ITA.19.87.047
9400,BEL,City,ff904d7013b2688c,75769.000,2011,2020.0,67296.70,2030,40.00,ff904d7013b2688c_2030,BEL.2.2.3.1_1
9403,ITA,City,ffa60dd141d5c943,83747.800,2010,2010.0,83747.80,2030,40.00,ffa60dd141d5c943_2030,ITA.15.63.073


In [26]:
(a["target_year"] > a["inv_year"]).all()

np.True_

In [27]:
0.008742 * 22806.7	* (2030-2011)

3788.1472566

In [28]:
(1 + (p["baseline_value"] - p["baseline_value"]*(1-p["target_value"]/100))/p["baseline_value"]) ** (1/(p["target_year"] - p["baseline_year"])) -1 

9393    0.008742
dtype: float64

In [29]:
climactor_ambition.loc[climactor_ambition["GDAM_id"] == "BEL.2.2.1.5_1", :]

,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,target_year,target_value,target_record,GDAM_id,target_year_emission,annualised_ambition_from_baseline,annualised_ambition,2019_emissions_for_total_ambition,2030_emissions,absolute_ambition


In [30]:
p["baseline_value"]*(1-p["target_value"]/100)

9393    13588.23186
dtype: float64

In [31]:
check = (
    climactor_ambition
    .groupby("GDAM_id")
    .agg(
        n_total_emissions=("total_emissions", "nunique"),
        n_baseline_value=("baseline_value", "nunique")
    )
)

result = check[check["n_total_emissions"] > 1]
(result["n_baseline_value"] == 1).all()
"ITA.19.84.002" in result.index.values
result.reset_index()

KeyError: "Label(s) ['total_emissions'] do not exist"

In [ ]:
climactor_ambition

,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,target_year,target_value,target_record,GDAM_id,target_year_emission,annualised_ambition_from_baseline,annualised_ambition,2019_emissions_for_total_ambition,2030_emissions,absolute_ambition
6,ESP,City,00752000db04c515,11423.000,2005,2014.0,2030,40.00,00752000db04c515_2030,ESP.09.08.08222,6853.80000,0.013550,0.012225,9.436959e+03,6.853800e+03,2583.159046
7,ITA,City,0083ef4ed71b869e,7739.785,2011,2011.0,2030,40.00,0083ef4ed71b869e_2030,ITA.19.86.003,4643.87100,0.017867,0.017867,6.700266e+03,4.643871e+03,2056.394823
10,BEL,City,00887dae5af7b824,48759.200,2011,2020.0,2030,40.00,00887dae5af7b824_2030,BEL.2.1.3.22_1,29255.52000,0.017867,0.030747,4.221042e+04,2.925552e+04,12954.903325
13,ITA,City,00b3f9e9c6778784,8965.400,2011,2011.0,2030,40.00,00b3f9e9c6778784_2030,ITA.19.84.007,5379.24000,0.017867,0.017867,7.761270e+03,5.379240e+03,2382.030269
17,ITA,City,00db591fd214bead,7876.169,2011,2011.0,2030,40.00,00db591fd214bead_2030,ITA.19.83.088,4725.70140,0.017867,0.017867,6.818332e+03,4.725701e+03,2092.630888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9387,USA,City,ff4e338346fcdebb,2597305.150,2019,2019.0,2050,100.00,ff4e338346fcdebb_2050,USA.16.77.1600000US1921000,0.00000,0.022611,0.022611,2.597305e+06,2.019588e+06,577717.230931
9393,ITA,City,ff8d794fe84c1e9d,22806.700,2011,2011.0,2050,40.42,ff8d794fe84c1e9d_2050,ITA.19.87.047,13588.23186,0.008742,0.008742,2.125960e+04,1.930223e+04,1957.372294
9400,BEL,City,ff904d7013b2688c,75769.000,2011,2020.0,2030,40.00,ff904d7013b2688c_2030,BEL.2.2.3.1_1,45461.40000,0.017867,0.028499,6.559258e+04,4.546140e+04,20131.176682
9403,ITA,City,ffa60dd141d5c943,83747.800,2010,2010.0,2030,40.00,ffa60dd141d5c943_2030,ITA.15.63.073,50248.68000,0.016966,0.016966,7.179440e+04,5.024868e+04,21545.717128


In [ ]:

climactor_ambition[climactor_ambition.groupby("GDAM_id")["inv_year"].transform("size") > 1]

,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,total_emissions,target_year,target_value,target_record,GDAM_id,target_year_emission,annualised_ambition_from_baseline,annualised_ambition,2019_emissions_for_total_ambition,2030_emissions,absolute_ambition
13,ITA,City,00b3f9e9c6778784,8965.4,2011,2011.0,8965.40,2030,40.0,00b3f9e9c6778784_2030,ITA.19.84.007,5379.240,0.017867,0.017867,7761.270269,5379.240000,2382.030269
14,ITA,City,00b3f9e9c6778784,8965.4,2011,2011.0,11575.00,2030,40.0,00b3f9e9c6778784_2030,ITA.19.84.007,5379.240,0.017867,0.022820,9623.156852,5379.240000,4243.916852
47,USA,City,026671f713ae1081,61893.0,2019,2019.0,63187.22,2030,62.7,026671f713ae1081_2030,USA.33.61.1600000US3632710,23086.089,0.045242,0.045688,63187.220000,23086.089000,40101.131000
48,USA,City,026671f713ae1081,61893.0,2019,2019.0,61893.00,2030,62.7,026671f713ae1081_2030,USA.33.61.1600000US3632710,23086.089,0.045242,0.045242,61893.000000,23086.089000,38806.911000
238,BEL,City,059234978a5cba53,154226.2,2011,2018.0,149763.60,2030,40.0,059234978a5cba53_2030,BEL.2.4.1.31_1,92535.720,0.017867,0.027335,145669.776921,92535.720000,53134.056921
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9210,ITA,City,f85db660df40fb07,6629.8,2011,2017.0,5147.70,2030,40.0,f85db660df40fb07_2030,ITA.19.84.002,3977.880,0.017867,0.015877,4985.539867,3977.880000,1007.659867
9273,USA,City,fb984a810dfc2407,301809.0,2018,2018.0,299729.68,2050,80.0,fb984a810dfc2407_2050,USA.39.48.1600000US4221648,60361.800,0.018538,0.018514,294180.627030,239519.888802,54660.738228
9274,USA,City,fb984a810dfc2407,301809.0,2018,2018.0,301809.00,2050,80.0,fb984a810dfc2407_2050,USA.39.48.1600000US4221648,60361.800,0.018538,0.018538,296214.044002,241109.150819,55104.893183
9316,USA,City,fd41dc8ea0d46728,228347.0,2018,2018.0,963987.00,2030,67.4,fd41dc8ea0d46728_2030,USA.22.5.1600000US2559105,74441.122,0.043870,0.055992,910011.131305,74441.122000,835570.009305


In [ ]:
climactor_ambition.loc[climactor_ambition["GDAM_id"] == "ITA.19.84.002", :]

,iso,entity_type,climactor_id,baseline_value,baseline_year,inv_year,target_year,target_value,target_record,GDAM_id,target_year_emission,annualised_ambition_from_baseline,annualised_ambition,2019_emissions_for_total_ambition,2030_emissions,absolute_ambition
9208,ITA,City,f85db660df40fb07,6629.8,2011,2017.0,2030,40.0,f85db660df40fb07_2030,ITA.19.84.002,3977.88,0.017867,0.017867,5739.361281,3977.88,1761.481281
